# ⚡ AP Physics C · 📷 PTZ/CMOS · 🔆 Photonic Si · 🔬 Laser · ☀️ Solar · 🤖 LLM Search

> **Chopped Griffiths format** — each §.x = one concept, one equation, one number, one plot.

| § | Topic | Boss equation |
|---|---|---|
| §1 | AP-C Kinematics / Newton | $F=ma$, work-energy, impulse |
| §2 | Gravity & Orbits | $g=-GM/r^2$, Hohmann ΔV, vis-viva |
| §3 | QM Conceptual | $\Delta x\,\Delta p\geq\hbar/2$, photoelectric, de Broglie |
| §4 | PTZ CMOS Sensor | pixel pitch, SNR, MTF, Bayer demosaic |
| §5 | Photonic Silicon | slab waveguide, ring resonator, MZI |
| §6 | Laser Rate Eqs | threshold, L-I curve, turn-on delay |
| §7 | Solar Cell IV | Shockley diode, fill factor, efficiency |
| §8 | LLM Auto Search | TF-IDF cosine match: question → solution stub |

In [ ]:
import numpy as np
import sympy as sp
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings, math, re
warnings.filterwarnings('ignore')
sp.init_printing(use_latex='mathjax')
from IPython.display import display, Math, Markdown
plt.rcParams.update({'font.size':10,'figure.dpi':100})
np.random.seed(42)
print('✦ AP Physics Opto — all modules loaded')

---
## §1 ⚡ AP Physics C — Kinematics · Newton · Work-Energy

**§1.1 Kinematic equations** (constant acceleration):
$$x = x_0 + v_0t + \tfrac{1}{2}at^2 \quad v^2 = v_0^2 + 2a\Delta x$$

**§1.2 Work-Energy theorem:**
$$W_{net} = \Delta KE = \tfrac{1}{2}mv^2 - \tfrac{1}{2}mv_0^2$$

**§1.3 Impulse-Momentum:**
$$J = \int F\,dt = \Delta p = m\Delta v$$

**§1.4 Drag + Newton — projectile with air resistance (ODE):**
$$m\ddot{x} = -b\dot{x}, \quad m\ddot{y} = -mg - b\dot{y}$$

In [ ]:
# §1 — AP-C Newton + kinematics
# ── SymPy: work-energy theorem symbolic ───────────────────────────
t_s, v0_s, a_s, m_s, b_s = sp.symbols('t v_0 a m b', positive=True)
x_s = v0_s*t_s + sp.Rational(1,2)*a_s*t_s**2
v_s = sp.diff(x_s, t_s)
KE  = sp.Rational(1,2)*m_s*v_s**2
W   = sp.integrate(m_s*a_s, (x_s, 0, x_s))   # W = F·dx = m·a·x (const F)
print('§1.1  x(t) =', sp.latex(x_s))
print('§1.2  KE   =', sp.latex(sp.expand(KE)))

# ── §1.4 Drag projectile ODE: RK45 ───────────────────────────────
m_v  = 0.5   # kg
b_v  = 0.1   # drag coefficient N·s/m
g_v  = 9.81
v0   = 30.0  # m/s
angles_deg = [15, 30, 45, 60, 75]

def drag_ode(t, y):
    x, yy, vx, vy = y
    speed = np.sqrt(vx**2 + vy**2)
    return [vx, vy, -b_v/m_v*vx, -g_v - b_v/m_v*vy]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ranges_drag = []; ranges_vac = []
for ang in angles_deg:               # loop: launch angles
    th = np.radians(ang)
    y0 = [0, 0, v0*np.cos(th), v0*np.sin(th)]
    sol = solve_ivp(drag_ode, [0,15], y0,
                    events=[lambda t,y,*a: y[1]],
                    dense_output=True, max_step=0.02)
    t_arr = np.linspace(0, sol.t[-1], 400)
    traj  = sol.sol(t_arr)
    axes[0].plot(traj[0], traj[1], lw=2, label=f'{ang}°')
    ranges_drag.append(traj[0,-1])
    # vacuum range
    T_vac = 2*v0*np.sin(th)/g_v
    ranges_vac.append(v0*np.cos(th)*T_vac)

axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_title('§1.4 Drag Projectile (b=0.1 N·s/m)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(bottom=0)

# Range comparison: drag vs vacuum
axes[1].bar(angles_deg, ranges_vac, width=3, alpha=0.5, label='Vacuum', color='blue')
axes[1].bar(angles_deg, ranges_drag, width=3, alpha=0.7, label='With drag', color='red')
axes[1].set_xlabel('Launch angle (°)'); axes[1].set_ylabel('Range (m)')
axes[1].set_title('§1.4 Range: vacuum vs drag'); axes[1].legend(); axes[1].grid(True,alpha=0.3)

# Energy: KE + PE vs time for 45° launch with drag
y0_45 = [0, 0, v0/np.sqrt(2), v0/np.sqrt(2)]
sol45  = solve_ivp(drag_ode, [0,10], y0_45, dense_output=True, max_step=0.02)
t45    = np.linspace(0, sol45.t[-1], 500)
tr45   = sol45.sol(t45)
KE45   = 0.5*m_v*(tr45[2]**2 + tr45[3]**2)
PE45   = m_v*g_v*tr45[1]
axes[2].plot(t45, KE45, 'r-', lw=2, label='KE')
axes[2].plot(t45, PE45, 'b-', lw=2, label='PE')
axes[2].plot(t45, KE45+PE45, 'k--', lw=1.5, label='Total (dissipated by drag)')
axes[2].set_xlabel('t (s)'); axes[2].set_ylabel('Energy (J)')
axes[2].set_title('§1.2 Work-Energy: 45° drag launch')
axes[2].legend(); axes[2].grid(True,alpha=0.3)
plt.suptitle('⚡ AP-C §1: Kinematics · Newton · Work-Energy', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §2 🌍 Gravity & Orbital Mechanics

**§2.1 Newton's law:** $g(r) = -GM/r^2$, $\Phi(r) = -GM/r$

**§2.2 Vis-viva:** $v^2 = GM\left(\dfrac{2}{r} - \dfrac{1}{a}\right)$

**§2.3 Hohmann transfer ΔV:**
$$\Delta V_1 = \sqrt{\frac{\mu}{r_1}}\left(\sqrt{\frac{2r_2}{r_1+r_2}}-1\right), \quad
\Delta V_2 = \sqrt{\frac{\mu}{r_2}}\left(1-\sqrt{\frac{2r_1}{r_1+r_2}}\right)$$

**§2.4 Escape velocity:** $v_{esc} = \sqrt{2GM/r}$

In [ ]:
# §2 — Gravity & Orbits
G  = 6.674e-11; M_E = 5.972e24; R_E = 6.371e6
mu = G * M_E

# §2.1 g(r) profile
r_arr = np.linspace(R_E, 10*R_E, 500)
g_arr = G*M_E / r_arr**2
phi_arr = -G*M_E / r_arr

# §2.3 Hohmann ΔV: LEO → GEO
r1 = R_E + 400e3      # LEO 400 km
r2 = R_E + 35786e3    # GEO
dv1 = np.sqrt(mu/r1)*(np.sqrt(2*r2/(r1+r2)) - 1)
dv2 = np.sqrt(mu/r2)*(1 - np.sqrt(2*r1/(r1+r2)))
t_transfer = np.pi*np.sqrt((r1+r2)**3 / (8*mu))
v_esc_LEO  = np.sqrt(2*mu/r1)

print(f'§2.3 LEO→GEO Hohmann:')
print(f'  ΔV₁ = {dv1/1e3:.3f} km/s  (at LEO)')
print(f'  ΔV₂ = {dv2/1e3:.3f} km/s  (at GEO)')
print(f'  ΔV_total = {(dv1+dv2)/1e3:.3f} km/s')
print(f'  Transfer time = {t_transfer/3600:.2f} hours')
print(f'§2.4 Escape velocity from LEO: {v_esc_LEO/1e3:.3f} km/s')

# Orbit simulation: circular + elliptical
def orbit_ode(t, y):
    x, yy, vx, vy = y
    r = np.sqrt(x**2+yy**2)
    F = -mu/r**3
    return [vx, vy, F*x, F*yy]

fig, axes = plt.subplots(1, 3, figsize=(15,4))

# Circular LEO
T_LEO  = 2*np.pi*np.sqrt(r1**3/mu)
v_LEO  = np.sqrt(mu/r1)
sol_c  = solve_ivp(orbit_ode, [0, T_LEO], [r1, 0, 0, v_LEO],
                   dense_output=True, max_step=T_LEO/200)
t_c    = np.linspace(0, T_LEO, 300)
tr_c   = sol_c.sol(t_c)
axes[0].plot(tr_c[0]/R_E, tr_c[1]/R_E, 'b-', lw=2, label='LEO (circular)')

# Hohmann transfer ellipse
a_h   = (r1+r2)/2
e_h   = (r2-r1)/(r2+r1)
T_h   = 2*np.pi*np.sqrt(a_h**3/mu)/2  # half period
sol_h = solve_ivp(orbit_ode, [0, T_h], [r1, 0, 0, v_LEO+dv1],
                  dense_output=True, max_step=T_h/200)
t_h   = np.linspace(0, T_h, 300)
tr_h  = sol_h.sol(t_h)
axes[0].plot(tr_h[0]/R_E, tr_h[1]/R_E, 'r--', lw=2, label='Hohmann transfer')

# GEO arc
T_GEO = 2*np.pi*np.sqrt(r2**3/mu)
v_GEO = np.sqrt(mu/r2)
sol_g  = solve_ivp(orbit_ode, [0, T_GEO], [r2, 0, 0, v_GEO],
                   dense_output=True, max_step=T_GEO/200)
t_g    = np.linspace(0, T_GEO, 400)
tr_g   = sol_g.sol(t_g)
axes[0].plot(tr_g[0]/R_E, tr_g[1]/R_E, 'g-', lw=2, label='GEO (circular)')
theta_E = np.linspace(0, 2*np.pi, 100)
axes[0].fill(np.cos(theta_E), np.sin(theta_E), color='cyan', alpha=0.4, label='Earth')
axes[0].set_aspect('equal'); axes[0].legend(fontsize=7)
axes[0].set_title('§2.3 Hohmann LEO→GEO'); axes[0].grid(True, alpha=0.2)
axes[0].set_xlabel('x (R_E)'); axes[0].set_ylabel('y (R_E)')

# g(r) and potential
axes[1].plot(r_arr/R_E, g_arr, 'b-', lw=2, label='g(r) m/s²')
axes[1].axvline(1, ls='--', color='k', lw=1, label='Earth surface')
axes[1].set_xlabel('r / R_E'); axes[1].set_ylabel('g (m/s²)')
axes[1].set_title('§2.1 Gravity field'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
ax1b = axes[1].twinx()
ax1b.plot(r_arr/R_E, phi_arr/1e6, 'r--', lw=1.5, label='Φ (MJ/kg)')
ax1b.set_ylabel('Φ (MJ/kg)', color='red')

# Vis-viva: v vs r for various semi-major axes
r_vis = np.linspace(R_E, 5*R_E, 300)
for a_vis in [R_E*1.5, R_E*2, R_E*3, R_E*4]:   # loop: semi-major axes
    v_vis = np.sqrt(mu*(2/r_vis - 1/a_vis))
    valid  = (2/r_vis - 1/a_vis) > 0
    axes[2].plot(r_vis[valid]/R_E, v_vis[valid]/1e3, lw=2,
                 label=f'a={a_vis/R_E:.1f}R_E')
axes[2].set_xlabel('r / R_E'); axes[2].set_ylabel('v (km/s)')
axes[2].set_title('§2.2 Vis-viva v(r)'); axes[2].legend(fontsize=8); axes[2].grid(True,alpha=0.3)
plt.suptitle('🌍 AP-C §2: Gravity · Orbits · Hohmann', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §3 🌌 QM Conceptual — Photoelectric · de Broglie · Uncertainty · Tunneling

**§3.1 Photoelectric:** $KE_{max} = hf - \phi$, stopping voltage $V_s = KE_{max}/e$

**§3.2 de Broglie:** $\lambda = h/p = h/mv$

**§3.3 Heisenberg:** $\Delta x\,\Delta p \geq \hbar/2$, $\Delta E\,\Delta t \geq \hbar/2$

**§3.4 Tunneling:** $T \approx e^{-2\kappa L}$, $\kappa = \sqrt{2m(V_0-E)}/\hbar$

**§3.5 Hydrogen energy levels:** $E_n = -13.6/n^2$ eV

In [ ]:
# §3 — QM Conceptual
h  = 6.626e-34; hbar = h/(2*np.pi)
c  = 3e8;  e_c = 1.602e-19; m_e = 9.109e-31
eV = e_c

# §3.1 Photoelectric effect
phi_metals = {'Cs':2.1,'Na':2.3,'Al':4.1,'Cu':4.7,'Pt':5.7}  # work function eV
f_arr = np.linspace(4e14, 2e15, 500)  # Hz

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for metal, phi in phi_metals.items():   # loop: metals
    KE = h*f_arr/eV - phi
    mask = KE > 0
    axes[0][0].plot(f_arr[mask]/1e14, KE[mask], lw=2, label=metal)
axes[0][0].set_xlabel('f (×10¹⁴ Hz)'); axes[0][0].set_ylabel('KE_max (eV)')
axes[0][0].set_title('§3.1 Photoelectric: KE_max vs f')
axes[0][0].legend(fontsize=8); axes[0][0].grid(True,alpha=0.3)

# §3.2 de Broglie: λ vs KE for electron, proton, neutron
particles = [('e⁻', m_e), ('p⁺', 1.673e-27), ('n⁰', 1.675e-27)]
KE_arr = np.logspace(-3, 4, 500) * eV   # 1 meV to 10 keV
for name, mass in particles:            # loop: particles
    p_arr  = np.sqrt(2*mass*KE_arr)
    lam_arr= h/p_arr
    axes[0][1].loglog(KE_arr/eV, lam_arr*1e9, lw=2, label=name)
axes[0][1].axhline(0.1, ls='--', color='k', lw=1, label='0.1 nm (atom)')
axes[0][1].set_xlabel('KE (eV)'); axes[0][1].set_ylabel('λ (nm)')
axes[0][1].set_title('§3.2 de Broglie wavelength')
axes[0][1].legend(fontsize=8); axes[0][1].grid(True,alpha=0.3)

# §3.3 Uncertainty principle: Δx·Δp ≥ ℏ/2
dx_arr = np.logspace(-15, -8, 300)  # 1 fm to 10 nm
dp_min = hbar/(2*dx_arr)
dE_min = dp_min**2/(2*m_e*eV)  # eV (electron)
axes[0][2].loglog(dx_arr*1e9, dp_min, 'b-', lw=2, label='Δp_min (kg·m/s)')
axes[0][2].set_xlabel('Δx (nm)'); axes[0][2].set_ylabel('Δp_min (kg·m/s)')
axes[0][2].set_title('§3.3 Heisenberg Δx·Δp ≥ ℏ/2')
axes[0][2].grid(True,alpha=0.3); axes[0][2].legend()
ax32b = axes[0][2].twinx()
ax32b.loglog(dx_arr*1e9, dE_min, 'r--', lw=1.5)
ax32b.set_ylabel('ΔE (eV, electron)', color='red')

# §3.4 Tunneling: T vs barrier width L for various E/V0
V0 = 5.0 * eV  # barrier height 5 eV
L_arr = np.linspace(0, 3e-9, 400)  # 0 to 3 nm
for E_ratio in [0.1, 0.3, 0.5, 0.7, 0.9]:  # loop: E/V0 ratios
    E  = E_ratio * V0
    kappa = np.sqrt(2*m_e*(V0-E))/hbar
    T = np.exp(-2*kappa*L_arr)
    axes[1][0].semilogy(L_arr*1e9, T, lw=2, label=f'E/V₀={E_ratio}')
axes[1][0].set_xlabel('L (nm)'); axes[1][0].set_ylabel('Transmission T')
axes[1][0].set_title('§3.4 Quantum Tunneling (V₀=5 eV)')
axes[1][0].legend(fontsize=8); axes[1][0].grid(True,alpha=0.3)
axes[1][0].set_ylim(1e-30, 1)

# §3.5 Hydrogen energy levels + Balmer series
n_arr  = np.arange(1, 9)
E_n    = -13.6 / n_arr**2  # eV
for n in n_arr:             # loop: energy levels
    axes[1][1].hlines(E_n[n-1], 0, 1, lw=2,
                       color=plt.cm.plasma(n/8), label=f'n={n}: {E_n[n-1]:.2f} eV')
# Balmer series: n→2
print('§3.5 Hydrogen Balmer series (n→2):')
for n in range(3,8):        # loop: Balmer lines
    dE   = 13.6*(1/4 - 1/n**2)
    lam  = h*c/(dE*eV)*1e9
    print(f'  n={n}→2: ΔE={dE:.3f} eV, λ={lam:.1f} nm')
    axes[1][1].annotate(f'n={n}→2\n{lam:.0f}nm',
                         xy=(0.5, -13.6/n**2), fontsize=7)
axes[1][1].set_xlim(0,1); axes[1][1].set_ylim(-15,1)
axes[1][1].set_ylabel('Energy (eV)'); axes[1][1].set_title('§3.5 Hydrogen levels')
axes[1][1].legend(fontsize=7, loc='upper right'); axes[1][1].grid(True,alpha=0.3)
axes[1][1].set_xticks([])

# §3.6 Wave function: particle in infinite square well
L_well = 1e-9  # 1 nm well
x_w = np.linspace(0, L_well, 400)
axes[1][2].set_title('§3.6 Infinite square well ψ_n(x)')
for n in range(1,5):        # loop: QM states
    psi = np.sqrt(2/L_well)*np.sin(n*np.pi*x_w/L_well)
    E_n_w = (n*np.pi*hbar)**2/(2*m_e*L_well**2)/eV
    axes[1][2].plot(x_w*1e9, psi/max(abs(psi))*0.8 + E_n_w,
                    lw=2, label=f'n={n}: {E_n_w:.2f} eV')
axes[1][2].set_xlabel('x (nm)'); axes[1][2].set_ylabel('E_n + ψ_n (eV)')
axes[1][2].legend(fontsize=8); axes[1][2].grid(True,alpha=0.3)
plt.suptitle('🌌 AP-C §3: QM Conceptual', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §4 📷 PTZ CMOS Sensor Physics

**§4.1 Pixel pitch & resolution:** $N_{px} = (W/p)\times(H/p)$, $p$ = pitch (μm)

**§4.2 SNR:** $\text{SNR} = Q_s / \sqrt{Q_s + Q_d + Q_r^2}$ (shot + dark + read noise)

**§4.3 MTF (Modulation Transfer Function):** diffraction limit $f_{cutoff} = 1/(\lambda\cdot f/\#)$

**§4.4 Bayer CFA:** RGGB demosaic — bilinear interpolation per channel

In [ ]:
# §4 — PTZ CMOS sensor physics
# §4.1 Pixel fill & resolution
sensor_W = 6.4e-3  # 6.4 mm width
sensor_H = 4.8e-3  # 4.8 mm height
pitches  = np.array([1.0,1.2,1.4,1.7,2.0,2.4,3.0]) * 1e-6  # μm
MPs      = (sensor_W/pitches) * (sensor_H/pitches) / 1e6

# §4.2 SNR model vs illuminance
lux_arr  = np.logspace(0, 5, 400)    # 1 to 100k lux
QE       = 0.70                       # quantum efficiency
t_exp    = 1/60                       # 60 fps
A_px     = (1.7e-6)**2                # 1.7 μm pixel
eta      = QE * t_exp * A_px * 1e17  # electrons per lux (rough model)
Q_s      = eta * lux_arr
Q_d      = 50.0    # dark electrons
Q_r      = 2.5     # read noise electrons RMS
SNR      = Q_s / np.sqrt(Q_s + Q_d + Q_r**2)

# §4.3 MTF diffraction limit vs f/#
f_num_arr = np.arange(1.4, 16, 0.5)
lam_vis   = 550e-9  # green
f_cut     = 1/(lam_vis * f_num_arr) / 1e3  # cy/mm

# §4.4 Bayer demosaic: synthetic RGGB
np.random.seed(7)
H_img, W_img = 64, 64
rgb_true = np.random.randint(50, 220, (H_img, W_img, 3), dtype=np.uint8)

def make_bayer(rgb):
    '''Simulate RGGB Bayer CFA sensor output.'''
    raw = np.zeros((H_img, W_img), dtype=np.uint8)
    raw[0::2, 0::2] = rgb[0::2, 0::2, 0]   # R
    raw[0::2, 1::2] = rgb[0::2, 1::2, 1]   # G
    raw[1::2, 0::2] = rgb[1::2, 0::2, 1]   # G
    raw[1::2, 1::2] = rgb[1::2, 1::2, 2]   # B
    return raw

def bayer_demosaic(raw):
    '''Bilinear demosaic of RGGB Bayer pattern.'''
    from scipy.ndimage import convolve
    out = np.zeros((H_img, W_img, 3), dtype=float)
    # R at R positions
    R_raw = np.zeros_like(raw, dtype=float)
    R_raw[0::2, 0::2] = raw[0::2, 0::2]
    G_raw = np.zeros_like(raw, dtype=float)
    G_raw[0::2, 1::2] = raw[0::2, 1::2]
    G_raw[1::2, 0::2] = raw[1::2, 0::2]
    B_raw = np.zeros_like(raw, dtype=float)
    B_raw[1::2, 1::2] = raw[1::2, 1::2]
    # Bilinear kernels
    k_full = np.array([[1,2,1],[2,4,2],[1,2,1]])/4.0
    k_diag = np.array([[1,0,1],[0,0,0],[1,0,1]])/4.0
    k_cross= np.array([[0,1,0],[1,0,1],[0,1,0]])/4.0
    out[:,:,0] = convolve(R_raw, k_full, mode='mirror')  # R bilinear
    out[:,:,1] = convolve(G_raw, k_cross, mode='mirror') + G_raw  # G
    out[:,:,2] = convolve(B_raw, k_full, mode='mirror')  # B bilinear
    return out.clip(0, 255).astype(np.uint8)

raw_img    = make_bayer(rgb_true)
demosaic_img = bayer_demosaic(raw_img)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Resolution vs pitch
axes[0][0].plot(pitches*1e6, MPs, 'bo-', ms=8, lw=2)
axes[0][0].set_xlabel('Pixel pitch (μm)'); axes[0][0].set_ylabel('Megapixels')
axes[0][0].set_title(f'§4.1 Resolution: {sensor_W*1e3:.1f}×{sensor_H*1e3:.1f} mm sensor')
axes[0][0].grid(True,alpha=0.3)
for p,mp in zip(pitches*1e6,MPs):
    if p in [1.0,1.7,3.0]: axes[0][0].annotate(f'{mp:.1f}MP',(p,mp),textcoords='offset points',xytext=(5,3),fontsize=8)

# SNR curve
axes[0][1].semilogx(lux_arr, 20*np.log10(SNR+1e-3), 'g-', lw=2)
axes[0][1].axhline(20, ls='--', color='orange', label='SNR 20 dB (usable)')
axes[0][1].axhline(40, ls='--', color='green',  label='SNR 40 dB (good)')
axes[0][1].set_xlabel('Illuminance (lux)'); axes[0][1].set_ylabel('SNR (dB)')
axes[0][1].set_title('§4.2 SNR vs Lux (1.7μm pixel, QE=70%)')
axes[0][1].legend(fontsize=8); axes[0][1].grid(True,alpha=0.3)

# MTF diffraction limit
axes[0][2].plot(f_num_arr, f_cut, 'r-o', ms=5, lw=2)
axes[0][2].set_xlabel('f/#'); axes[0][2].set_ylabel('Diffraction cutoff (cy/mm)')
axes[0][2].set_title(f'§4.3 MTF diffraction limit @ λ={lam_vis*1e9:.0f} nm')
axes[0][2].grid(True,alpha=0.3)
for fn in [2.8, 5.6, 11.0]:
    fc = 1/(lam_vis*fn)/1e3
    axes[0][2].annotate(f'f/{fn:.0f}
{fc:.0f}cy/mm',(fn,fc),fontsize=7,
                         textcoords='offset points',xytext=(3,-15))

# Bayer demo
axes[1][0].imshow(raw_img, cmap='gray'); axes[1][0].set_title('§4.4 Raw Bayer (RGGB)')
axes[1][0].axis('off')
axes[1][1].imshow(rgb_true); axes[1][1].set_title('§4.4 Ground truth RGB')
axes[1][1].axis('off')
axes[1][2].imshow(demosaic_img); axes[1][2].set_title('§4.4 Bilinear demosaic')
axes[1][2].axis('off')

plt.suptitle('📷 §4: PTZ CMOS Sensor Physics', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'§4.1  1.7 μm pixel: {(sensor_W/1.7e-6)*(sensor_H/1.7e-6)/1e6:.1f} MP')
print(f'§4.2  SNR at 100 lux: {20*np.log10(SNR[np.argmin(abs(lux_arr-100))]+1e-3):.1f} dB')
print(f'§4.3  Diffraction limit at f/2.8: {1/(lam_vis*2.8)/1e3:.0f} cy/mm')

---
## §5 🔆 Photonic Silicon — Slab Waveguide · Ring Resonator · MZI

**§5.1 Slab waveguide dispersion:**
$k_0 n_1 \cos\theta_1 = m\pi/d$ (eigenvalue), $n_{eff} = n_1 \sin\theta_1$

**§5.2 Ring resonator:**
$\lambda_{res} = n_{eff} L / m$, $Q = \lambda_{res}/\Delta\lambda$, through-port: $T = (a-r)^2/(1-ar)^2$ at resonance (all-pass)

**§5.3 MZI transfer:**
$T = \cos^2(\Delta\phi/2)$, $\Delta\phi = (2\pi/\lambda)\Delta n_{eff}\cdot L$

In [ ]:
# §5 — Photonic Silicon
lam_arr = np.linspace(1500e-9, 1600e-9, 2000)  # telecom C-band
n_Si    = 3.48; n_SiO2 = 1.44; n_air = 1.0
wl0     = 1550e-9; k0_0   = 2*np.pi/wl0

# §5.1 Slab waveguide: TE effective index (height h=220 nm Si on SiO2)
h_wg = 220e-9
def slab_TE_neff(lam, h, n_core, n_sub, n_clad=1.0):
    '''Numerical TE dispersion: transcendental equation root find.'''
    k0 = 2*np.pi/lam
    def eq(neff):
        if neff < max(n_sub,n_clad) or neff > n_core: return 1e10
        kx  = k0*np.sqrt(n_core**2 - neff**2)
        gam_sub = k0*np.sqrt(neff**2 - n_sub**2)
        gam_cl  = k0*np.sqrt(neff**2 - n_clad**2)
        return np.tan(kx*h) - (gam_sub + gam_cl)/(kx*(1 - gam_sub*gam_cl/kx**2))
    try:
        return brentq(eq, max(n_sub,n_clad)+1e-6, n_core-1e-6)
    except:
        return np.nan

neff_arr = np.array([slab_TE_neff(lam, h_wg, n_Si, n_SiO2) for lam in lam_arr])
# Group index: ng = neff - λ * d(neff)/dλ
dneff = np.gradient(neff_arr, lam_arr)
ng_arr = neff_arr - lam_arr*dneff
print(f'§5.1  n_eff @ 1550nm = {neff_arr[len(lam_arr)//2]:.4f}')
print(f'§5.1  n_g   @ 1550nm = {ng_arr[len(lam_arr)//2]:.4f}')

# §5.2 Ring resonator: all-pass through-port transmission
R_ring  = 10e-6   # radius 10 μm
L_ring  = 2*np.pi*R_ring
r_coup  = 0.95    # self-coupling coefficient
a_loss  = np.exp(-0.5 * 3.0 * L_ring / (10/np.log(10)))  # 3 dB/cm propagation loss
T_ring  = np.zeros(len(lam_arr))
for i, lam in enumerate(lam_arr):    # loop: wavelengths
    neff_i = neff_arr[i] if not np.isnan(neff_arr[i]) else neff_arr[1000]
    phi    = 2*np.pi*neff_i*L_ring/lam
    T_ring[i] = (a_loss**2 - 2*r_coup*a_loss*np.cos(phi) + r_coup**2) /                 (1 - 2*r_coup*a_loss*np.cos(phi) + (r_coup*a_loss)**2)
# FSR and Q
res_dips = np.where((T_ring[:-1]>T_ring[1:]) & (T_ring[1:]<T_ring[:-2]))[0]
if len(res_dips) >= 2:
    FSR = abs(lam_arr[res_dips[1]] - lam_arr[res_dips[0]])
    lam_res = lam_arr[res_dips[0]]
    BW  = FSR * (1-r_coup*a_loss)**2 / (np.pi*(r_coup*a_loss)**0.5)
    Q   = lam_res / (FSR*(1-r_coup**2*a_loss**2)/(np.pi*r_coup*a_loss))
    print(f'§5.2  Ring FSR = {FSR*1e9:.2f} nm, Q ≈ {lam_res/BW:.0f}')

# §5.3 MZI: ΔL sweep → transmission
delta_L_arr = np.linspace(0, 100e-6, 500)  # 0 to 100 μm
neff_0      = neff_arr[len(neff_arr)//2]
lam_mzi     = 1550e-9
phi_MZI     = 2*np.pi*neff_0*delta_L_arr/lam_mzi
T_MZI       = np.cos(phi_MZI/2)**2

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0][0].plot(lam_arr*1e9, neff_arr, 'b-', lw=2, label='n_eff TE₀')
axes[0][0].plot(lam_arr*1e9, ng_arr,   'r--', lw=2, label='n_g TE₀')
axes[0][0].set_xlabel('λ (nm)'); axes[0][0].set_ylabel('Index')
axes[0][0].set_title(f'§5.1 Si slab waveguide h={h_wg*1e9:.0f} nm')
axes[0][0].legend(); axes[0][0].grid(True,alpha=0.3)

axes[0][1].plot(lam_arr*1e9, 10*np.log10(T_ring+1e-20), 'g-', lw=1.5)
axes[0][1].set_xlabel('λ (nm)'); axes[0][1].set_ylabel('Transmission (dB)')
axes[0][1].set_title(f'§5.2 Ring resonator (R={R_ring*1e6:.0f} μm, r={r_coup})')
axes[0][1].grid(True,alpha=0.3); axes[0][1].set_ylim(-25,1)

axes[1][0].plot(delta_L_arr*1e6, T_MZI, 'm-', lw=2)
axes[1][0].set_xlabel('ΔL (μm)'); axes[1][0].set_ylabel('Transmission')
axes[1][0].set_title('§5.3 MZI transfer function @ 1550 nm')
axes[1][0].grid(True,alpha=0.3)

# 2D mode profile: Gaussian approximation for Si wire
x_mode = np.linspace(-1e-6, 1e-6, 200)
y_mode = np.linspace(-0.5e-6, 0.5e-6, 200)
X_m, Y_m = np.meshgrid(x_mode, y_mode)
wx, wy = 0.35e-6, 0.18e-6  # mode field half-widths
E_mode = np.exp(-(X_m/wx)**2 - (Y_m/wy)**2)
im_m = axes[1][1].imshow(E_mode**2, cmap='hot', origin='lower',
                           extent=[-1,1,-0.5,0.5], aspect='auto')
axes[1][1].set_xlabel('x (μm)'); axes[1][1].set_ylabel('y (μm)')
axes[1][1].set_title('§5.1 Si wire mode |E|² (450×220 nm)')
plt.colorbar(im_m, ax=axes[1][1])
plt.suptitle('🔆 §5: Photonic Silicon · Waveguide · Ring · MZI', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §6 🔬 Laser Rate Equations — Threshold · L-I Curve · Turn-on Delay

$$\frac{dN}{dt} = \frac{I}{eV} - \frac{N}{\tau_n} - v_g g_0 \frac{N-N_{tr}}{1+\varepsilon S}S$$

$$\frac{dS}{dt} = \Gamma v_g g_0 \frac{N-N_{tr}}{1+\varepsilon S}S - \frac{S}{\tau_p} + \Gamma\beta\frac{N}{\tau_n}$$

Above threshold: $I_{th} = eV N_{th}/\tau_n$, slope efficiency $\eta_d = dP/dI$.

In [ ]:
# §6 — Semiconductor Laser Rate Equations
# Parameters (typical 850 nm GaAs laser)
V_act  = 3e-17    # active volume m³
Gamma  = 0.3      # confinement factor
vg     = 0.75*3e8 # group velocity
g0     = 1e-12    # gain coefficient m³/s
N_tr   = 1.5e24   # transparency carrier density m⁻³
eps    = 2e-23    # gain compression factor m³
tau_n  = 2e-9     # carrier lifetime s
tau_p  = 2e-12    # photon lifetime s
beta   = 1e-4     # spontaneous emission factor
e_c    = 1.602e-19
hnu    = 1.42*e_c # photon energy (850 nm)

def laser_rate(t, y, I_bias):
    N, S = y
    N = max(N, 0); S = max(S, 1)
    g    = vg*g0*(N-N_tr)/(1+eps*S)
    dN   = I_bias/(e_c*V_act) - N/tau_n - g*S
    dS   = Gamma*g*S - S/tau_p + Gamma*beta*N/tau_n
    return [dN, dS]

# §6.1 Threshold current estimate
I_th = e_c*V_act*(N_tr + 1/(Gamma*vg*g0*tau_p)) / tau_n
print(f'§6.1 Threshold current I_th ≈ {I_th*1e3:.2f} mA')

# §6.2 L-I curve (steady state)
I_arr   = np.linspace(0, 4*I_th, 60)
P_out   = []
for I_v in I_arr:                      # loop: bias current
    sol_ss = solve_ivp(laser_rate, [0, 20e-9], [N_tr, 1e10],
                       args=(I_v,), dense_output=True, max_step=5e-12)
    S_ss = sol_ss.y[1,-1]
    P_out.append(S_ss * V_act * hnu / tau_p)
P_out = np.array(P_out)

# §6.3 Turn-on transient at 3×I_th
sol_ton = solve_ivp(laser_rate, [0, 6e-9], [N_tr, 1e5],
                    args=(3*I_th,), dense_output=True, max_step=2e-12)
t_ton   = np.linspace(0, 6e-9, 2000)
y_ton   = sol_ton.sol(t_ton)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(I_arr*1e3, P_out*1e3, 'b-', lw=2)
axes[0].axvline(I_th*1e3, ls='--', color='r', lw=1.5, label=f'I_th={I_th*1e3:.2f} mA')
axes[0].set_xlabel('I (mA)'); axes[0].set_ylabel('P_out (mW)')
axes[0].set_title('§6.2 L-I curve'); axes[0].legend(); axes[0].grid(True,alpha=0.3)

# Slope efficiency
above = I_arr > 1.3*I_th
if above.sum() > 3:
    slope_fit = np.polyfit(I_arr[above], P_out[above], 1)
    eta_d     = slope_fit[0]/e_c * hnu
    print(f'§6.2 Slope efficiency η_d = {slope_fit[0]*1e3:.1f} mW/mA')
    axes[0].plot(I_arr[above]*1e3, np.polyval(slope_fit, I_arr[above])*1e3,
                 'r--', lw=1, label=f'η_d={slope_fit[0]*1e3:.1f} mW/mA')
    axes[0].legend(fontsize=8)

# Turn-on transient
ax_N = axes[1]
ax_S = ax_N.twinx()
ax_N.plot(t_ton*1e9, y_ton[0]/N_tr, 'b-', lw=2, label='N/N_tr')
ax_S.plot(t_ton*1e9, y_ton[1]/max(y_ton[1])*0.9, 'r-', lw=1.5, label='S (norm)', alpha=0.8)
ax_N.set_xlabel('t (ns)'); ax_N.set_ylabel('N/N_tr', color='blue')
ax_S.set_ylabel('S (normalized)', color='red')
axes[1].set_title('§6.3 Turn-on transient @ 3×I_th')
ax_N.legend(loc='upper left',fontsize=8); ax_S.legend(loc='upper right',fontsize=8)
ax_N.grid(True,alpha=0.3)

# Small-signal modulation bandwidth
# fr = (1/2π)√(vg*g*S/τp) ≈ (1/2π)√(gain×photons/τp)
I_mod   = np.linspace(1.1*I_th, 4*I_th, 50)
f_r     = np.zeros(len(I_mod))
for j, I_v in enumerate(I_mod):       # loop: current points
    sol_j = solve_ivp(laser_rate, [0, 20e-9], [N_tr,1e10],
                      args=(I_v,), dense_output=True, max_step=5e-12)
    N_j = sol_j.y[0,-1]; S_j = sol_j.y[1,-1]
    g_j = vg*g0*(N_j-N_tr)/(1+eps*S_j)
    fr_j= np.sqrt(g_j*S_j/tau_p) / (2*np.pi)
    f_r[j] = fr_j/1e9  # GHz
axes[2].plot(I_mod*1e3, f_r, 'g-o', ms=4, lw=2)
axes[2].set_xlabel('I (mA)'); axes[2].set_ylabel('f_r (GHz)')
axes[2].set_title('§6.4 Relaxation oscillation freq')
axes[2].grid(True,alpha=0.3)
plt.suptitle('🔬 §6: Laser Rate Equations', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §7 ☀️ Solar Cell — Shockley Diode · IV Curve · Fill Factor · Efficiency

$$I = I_{ph} - I_0\left(e^{qV/nkT}-1\right) - V/R_{sh}$$

**Fill Factor:** $FF = P_{max}/(V_{oc}\cdot I_{sc})$

**Efficiency:** $\eta = P_{max}/P_{in}$, $P_{in} = 1000$ W/m² (AM1.5)

In [ ]:
# §7 — Solar cell IV curve (Shockley single-diode model)
kT   = 1.381e-23 * 298  # room temperature
q    = 1.602e-19
AM15 = 1000.0   # W/m²
A    = 1.0      # cm² cell area (normalized)

# Parameters (crystalline Si)
I_ph = 0.035    # A (photocurrent)
I_0  = 1e-10    # A (dark saturation)
n_id = 1.2      # ideality factor
R_s  = 0.5      # series resistance Ω
R_sh = 300.0    # shunt resistance Ω

def cell_IV(V, I_ph, I_0, n_id, R_s, R_sh):
    '''Implicit I-V: solve for I given V (Newton iteration).'''
    I = I_ph - I_0*(np.exp(q*V/(n_id*kT)) - 1)
    for _ in range(20):   # loop: Newton iterations
        F  = I - I_ph + I_0*(np.exp(q*(V+I*R_s)/(n_id*kT))-1) + (V+I*R_s)/R_sh
        dF = 1 + I_0*q*R_s/(n_id*kT)*np.exp(q*(V+I*R_s)/(n_id*kT)) + R_s/R_sh
        I  = I - F/dF
    return I

V_arr  = np.linspace(0, 0.65, 500)
I_arr2 = np.array([cell_IV(v, I_ph, I_0, n_id, R_s, R_sh) for v in V_arr])
P_arr  = V_arr * I_arr2.clip(0)

V_oc   = V_arr[np.argmin(abs(I_arr2))]
I_sc   = float(cell_IV(0, I_ph, I_0, n_id, R_s, R_sh))
P_max  = P_arr.max()
V_mp   = V_arr[P_arr.argmax()]
I_mp   = I_arr2[P_arr.argmax()]
FF     = P_max / (V_oc * I_sc)
eta    = P_max / (AM15 * A * 1e-4) * 100  # % (A in cm²)

print(f'§7  I_sc = {I_sc*1e3:.2f} mA')
print(f'§7  V_oc = {V_oc:.3f} V')
print(f'§7  P_max= {P_max*1e3:.2f} mW,  FF = {FF:.3f}')
print(f'§7  η    = {eta:.1f}%')

# Temperature coefficient: η vs T
T_arr_C = np.arange(0, 80, 5)
eta_T   = []
for T_C in T_arr_C:              # loop: temperatures
    kT_t = 1.381e-23*(T_C+273)
    I0_t  = I_0 * np.exp(q*1.12/(n_id*kT_t))  # rough Eg temperature shift
    Iph_t = I_ph*(1 - 4e-4*(T_C-25))           # Isc temp coefficient
    V_arr_t = np.linspace(0, 0.70, 400)
    I_arr_t = np.array([cell_IV(v, Iph_t, I0_t, n_id, R_s, R_sh) for v in V_arr_t])
    P_t  = (V_arr_t * I_arr_t.clip(0)).max()
    eta_T.append(P_t / (AM15*A*1e-4)*100)

# Parameter sweep: η vs R_s
R_s_arr  = np.linspace(0, 5, 40)
eta_Rs   = []
for rs in R_s_arr:               # loop: series resistance
    I_rs = np.array([cell_IV(v, I_ph, I_0, n_id, rs, R_sh) for v in V_arr])
    P_rs = (V_arr * I_rs.clip(0)).max()
    eta_Rs.append(P_rs/(AM15*A*1e-4)*100)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
ax_i = axes[0]
ax_p = ax_i.twinx()
ax_i.plot(V_arr, I_arr2*1e3, 'b-', lw=2, label='I (mA)')
ax_p.plot(V_arr, P_arr*1e3,  'r--', lw=2, label='P (mW)')
ax_i.axvline(V_mp, ls=':', color='gray'); ax_i.axhline(0, color='k', lw=0.5)
ax_i.fill_betweenx([0,I_mp*1e3],[0,0],[V_mp,V_mp],alpha=0.15,color='gold',label=f'P_max={P_max*1e3:.1f}mW')
ax_i.set_xlabel('V (V)'); ax_i.set_ylabel('I (mA)', color='blue')
ax_p.set_ylabel('P (mW)', color='red')
ax_i.set_title(f'§7 IV curve  FF={FF:.3f}  η={eta:.1f}%')
ax_i.legend(fontsize=8,loc='upper right'); ax_p.legend(fontsize=8,loc='center right')
ax_i.grid(True,alpha=0.3)

axes[1].plot(T_arr_C, eta_T, 'ro-', ms=5, lw=2)
axes[1].set_xlabel('Temperature (°C)'); axes[1].set_ylabel('η (%)')
axes[1].set_title('§7 Efficiency vs Temperature'); axes[1].grid(True,alpha=0.3)
tc = np.polyfit(T_arr_C, eta_T, 1)[0]
axes[1].annotate(f'TC = {tc:.3f} %/°C', (40, np.interp(40,T_arr_C,eta_T)), fontsize=9,
                  textcoords='offset points', xytext=(5,-15))

axes[2].plot(R_s_arr, eta_Rs, 'g-o', ms=5, lw=2)
axes[2].axvline(R_s, ls='--', color='r', lw=1.5, label=f'R_s={R_s}Ω (nominal)')
axes[2].set_xlabel('Series resistance R_s (Ω)'); axes[2].set_ylabel('η (%)')
axes[2].set_title('§7 η vs series resistance'); axes[2].legend(fontsize=8); axes[2].grid(True,alpha=0.3)
plt.suptitle('☀️ §7: Solar Cell IV · Fill Factor · Efficiency', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## §8 🤖 LLM Auto Search & Match — Physics Q→Solution Stub

**Method:** TF-IDF vectorize a library of physics problem stubs.
Type any question → cosine similarity → top-3 matching solutions.

This is the local "poor man's RAG" for AP Physics C:
- Chunk Griffiths/Serway solution steps into a corpus
- Embed with TF-IDF (no GPU needed)
- Match at query time: $\text{sim}(q,d) = \frac{q\cdot d}{|q||d|}$

In [ ]:
# §8 — LLM Auto Search & Match (TF-IDF Physics RAG)
# ── Physics solution stub corpus ──────────────────────────────────
CORPUS = [
    # Mechanics
    {"id":"M01","topic":"Kinematics","eq":"x = x0 + v0t + 0.5at^2",
     "stub":"Given initial velocity v0 and acceleration a, find position at time t. "
            "Use kinematic equations. Displacement Δx = v0t + 0.5at². "
            "Check units: meters, seconds, m/s²."},
    {"id":"M02","topic":"Work-Energy","eq":"W = ΔKE = 0.5mv^2 - 0.5mv0^2",
     "stub":"Work-energy theorem: net work equals change in kinetic energy. "
            "Integrate F·dx over path. Conservative forces have potential energy U."},
    {"id":"M03","topic":"Momentum","eq":"J = FΔt = Δp = mΔv",
     "stub":"Impulse equals change in momentum. For collisions: elastic conserves KE, "
            "inelastic does not. Use conservation of momentum: p_before = p_after."},
    {"id":"M04","topic":"Circular motion","eq":"a_c = v^2/r = ω^2 r",
     "stub":"Centripetal acceleration directed inward. Net radial force = mv^2/r. "
            "For vertical circle: at top N+mg = mv^2/r, at bottom N-mg = mv^2/r."},
    {"id":"M05","topic":"Rotational inertia","eq":"I = Σmr^2, τ = Iα",
     "stub":"Moment of inertia depends on axis. Parallel axis: I = I_cm + Md^2. "
            "Torque τ = r×F. Angular momentum L = Iω conserved if τ_net=0."},
    # Gravity
    {"id":"G01","topic":"Newton gravity","eq":"F = GMm/r^2",
     "stub":"Universal gravitation. g = GM/r^2. Orbital speed v = sqrt(GM/r). "
            "Period T = 2π sqrt(r^3/GM). Geosynchronous orbit r ≈ 42,164 km."},
    {"id":"G02","topic":"Escape velocity","eq":"v_esc = sqrt(2GM/r)",
     "stub":"Set KE = PE: 0.5mv^2 = GMm/r. Escape velocity is sqrt(2GM/r). "
            "From Earth surface: v_esc ≈ 11.2 km/s."},
    {"id":"G03","topic":"Hohmann transfer","eq":"ΔV1 = sqrt(μ/r1)(sqrt(2r2/(r1+r2))-1)",
     "stub":"Two-burn maneuver to change orbits. Burn 1 at periapsis raises apoapsis, "
            "Burn 2 at apoapsis circularizes. Total ΔV = ΔV1 + ΔV2."},
    # E&M
    {"id":"E01","topic":"Coulomb law","eq":"F = kq1q2/r^2",
     "stub":"Electric force between charges. k = 9×10^9 N·m²/C². "
            "Electric field E = F/q = kQ/r^2. Superposition for multiple charges."},
    {"id":"E02","topic":"Gauss law","eq":"∮E·dA = Q_enc/ε0",
     "stub":"Gauss's law: flux through closed surface = enclosed charge/ε0. "
            "Use for symmetric distributions: spherical, cylindrical, planar."},
    {"id":"E03","topic":"RC circuits","eq":"V(t) = V0 e^(-t/RC)",
     "stub":"Charging: V(t) = V0(1-e^(-t/RC)). Discharging: V(t) = V0 e^(-t/RC). "
            "Time constant τ = RC. At t=τ, voltage drops to 37% of initial."},
    {"id":"E04","topic":"Inductance","eq":"V_L = L dI/dt, τ=L/R",
     "stub":"Inductor opposes current change. RL circuit: I(t) = I_max(1-e^(-t/τ)). "
            "Energy stored: U = 0.5LI^2. Impedance Z_L = jωL."},
    # QM
    {"id":"Q01","topic":"Photoelectric effect","eq":"KE_max = hf - φ",
     "stub":"Einstein 1905: photon energy E=hf absorbed by electron. "
            "KE_max = hf - φ (work function). Stopping voltage V_s = KE_max/e. "
            "Threshold frequency f0 = φ/h."},
    {"id":"Q02","topic":"de Broglie","eq":"λ = h/p = h/mv",
     "stub":"Matter waves: wavelength λ = h/p. For electron at 100 eV: "
            "p = sqrt(2mKE), λ = h/p ≈ 0.12 nm (X-ray scale)."},
    {"id":"Q03","topic":"Heisenberg uncertainty","eq":"Δx·Δp ≥ ℏ/2",
     "stub":"Position-momentum uncertainty: ΔxΔp ≥ ℏ/2. "
            "Energy-time: ΔEΔt ≥ ℏ/2. Fundamental quantum limit, not measurement error."},
    {"id":"Q04","topic":"Hydrogen atom","eq":"E_n = -13.6/n^2 eV",
     "stub":"Bohr model energy levels. Ground state n=1: E=-13.6 eV. "
            "Lyman: n→1 UV. Balmer: n→2 visible. Paschen: n→3 IR."},
    {"id":"Q05","topic":"Quantum tunneling","eq":"T ≈ exp(-2κL)",
     "stub":"Particle penetrates classically forbidden region. "
            "κ = sqrt(2m(V0-E))/ℏ. Tunnel diode, STM, nuclear fusion all exploit tunneling."},
    # Optics / Photonics
    {"id":"P01","topic":"Snell law TIR","eq":"n1 sin θ1 = n2 sin θ2",
     "stub":"Refraction: n1 sinθ1 = n2 sinθ2. Total internal reflection when "
            "sinθ_c = n2/n1. Fiber optic NA = sqrt(n1^2-n2^2)."},
    {"id":"P02","topic":"Laser threshold","eq":"I_th = eV N_th / τ_n",
     "stub":"Lasing threshold: gain = loss. G_th = αi + (1/2L)ln(1/R1R2). "
            "Above threshold: output power P = η_d(I-I_th)hν/e."},
    {"id":"P03","topic":"Solar cell efficiency","eq":"η = P_max / P_in",
     "stub":"Shockley-Queisser limit ≈ 33% for single junction (Eg≈1.4 eV). "
            "Fill factor FF = Pmax/(Voc×Isc). Series resistance degrades FF."},
    {"id":"P04","topic":"CMOS SNR","eq":"SNR = Qs/sqrt(Qs+Qd+σr^2)",
     "stub":"Shot noise limited at high light. Read noise dominates at low light. "
            "Full well capacity sets dynamic range. Larger pixels → better SNR."},
]

# Build TF-IDF index
texts = [f"{d['topic']} {d['eq']} {d['stub']}" for d in CORPUS]
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words='english')
tfidf_mat  = vectorizer.fit_transform(texts)

def physics_search(query, top_k=3):
    q_vec  = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_mat)[0]
    top_idx= scores.argsort()[::-1][:top_k]
    results= []
    for idx in top_idx:
        d = CORPUS[idx]
        results.append({'id':d['id'],'topic':d['topic'],'score':scores[idx],
                        'eq':d['eq'],'stub':d['stub']})
    return results

# ── Demo queries ──────────────────────────────────────────────────
queries = [
    "how do I find the escape velocity from Earth surface",
    "what is the stopping voltage in photoelectric experiment",
    "RC circuit time constant voltage decay",
    "Bayer pattern demosaic camera sensor noise",
    "hydrogen energy level Balmer series wavelength",
    "ring resonator quality factor free spectral range",
]

print('🤖 Physics Auto-Search (TF-IDF cosine similarity)
')
print('='*70)
for q in queries:                        # loop: demo queries
    print(f'
❓ Query: "{q}"')
    for r in physics_search(q, top_k=2): # loop: results
        print(f'  [{r["id"]}] {r["topic"]:20s}  score={r["score"]:.3f}')
        print(f'       eq: {r["eq"]}')
        print(f'       → {r["stub"][:80]}...')

# ── Visualization: TF-IDF similarity heatmap ──────────────────────
sim_matrix = cosine_similarity(tfidf_mat).round(2)
topics     = [d['topic'][:12] for d in CORPUS]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
im_sim = axes[0].imshow(sim_matrix, cmap='viridis', vmin=0, vmax=1)
axes[0].set_xticks(range(len(CORPUS))); axes[0].set_yticks(range(len(CORPUS)))
axes[0].set_xticklabels(topics, rotation=90, fontsize=6)
axes[0].set_yticklabels(topics, fontsize=6)
axes[0].set_title(f'TF-IDF pairwise similarity ({len(CORPUS)} solutions)')
plt.colorbar(im_sim, ax=axes[0])

# Score bar chart for first query
q_demo = "escape velocity satellite orbital mechanics"
results_demo = physics_search(q_demo, top_k=len(CORPUS))
ids_d  = [r['id']   for r in results_demo]
scr_d  = [r['score'] for r in results_demo]
colors_d = ['green' if s>0.1 else 'steelblue' for s in scr_d]
axes[1].barh(range(len(ids_d)), scr_d, color=colors_d, alpha=0.8)
axes[1].set_yticks(range(len(ids_d)))
axes[1].set_yticklabels([f"{r['id']} {r['topic'][:15]}" for r in results_demo], fontsize=7)
axes[1].set_xlabel('Cosine similarity score')
axes[1].set_title(f'Query: "{q_demo}"')
axes[1].axvline(0.1, ls='--', color='red', lw=1, label='threshold')
axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)
plt.suptitle('🤖 §8: LLM Auto Search & Match — Physics RAG', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'\nCorpus: {len(CORPUS)} solution stubs · {tfidf_mat.shape[1]} TF-IDF features')